# 60 Eval Export Smoke

Runs final evaluation and export smoke checks.

Must cover:
- identity correctness (name/org)
- unprompted self-branding rate on neutral prompts
- vision hallucination on no-image prompts
- multimodal correctness on image prompts
- GGUF export smoke with q8_0 + q4 candidate
- template parity + no garbled outputs


In [ ]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import yaml

from lumis1.export_smoke import run_export_smoke
from lumis1.hashing import sha256_file
from lumis1.run_evidence import (
    assess_eval_export_status,
    create_run_evidence_tree,
    write_run_status,
    write_run_summary,
)

REPO_ROOT = Path.cwd().resolve()
REPORTS = REPO_ROOT / "workspace" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
dpo_cfg = yaml.safe_load((REPO_ROOT / "configs" / "train_dpo.yaml").read_text(encoding="utf-8"))

RUN_EVAL = False
RUN_EXPORT = False
RUN_ID = "manual-eval-export"

MODEL_DIR = REPO_ROOT / dpo_cfg["outputs"]["run_dir"]
GGUF_DIR = REPO_ROOT / "workspace" / "runs" / "export_gguf"

identity_prompts = [
    "What is your name?",
    "Who created you?",
]
neutral_prompts = [
    "Summarize this paragraph in one sentence.",
    "What is the weather process for rain formation?",
    "Rewrite this sentence in simpler English.",
]
no_image_prompts = [
    "Describe the image.",
    "What objects are visible here?",
]

results = {
    "run_eval": RUN_EVAL,
    "run_export": RUN_EXPORT,
    "run_id": RUN_ID,
    "checks": {
        "identity_correctness": {"status": "skipped"},
        "self_branding_rate": {"status": "skipped"},
        "vision_hallucination_on_no_image": {"status": "skipped"},
        "multimodal_correctness": {"status": "skipped"},
    },
}

run_paths = None
if RUN_EVAL or RUN_EXPORT:
    run_paths = create_run_evidence_tree(REPO_ROOT, RUN_ID)
    (run_paths["config_snapshot"] / "eval_export_inputs.json").write_text(
        json.dumps(
            {
                "model_dir": str(MODEL_DIR),
                "gguf_dir": str(GGUF_DIR),
                "run_eval": RUN_EVAL,
                "run_export": RUN_EXPORT,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    (run_paths["commands"] / "notebook_60_invocation.txt").write_text(
        "Notebook 60 manual execution for eval/export smoke.\n"
        f"run_eval={RUN_EVAL}\n"
        f"run_export={RUN_EXPORT}\n"
        f"model_dir={MODEL_DIR}\n"
        f"gguf_dir={GGUF_DIR}\n",
        encoding="utf-8",
    )
    (run_paths["environment"] / "runtime.json").write_text(
        json.dumps(
            {
                "python": sys.version,
                "platform": platform.platform(),
                "cwd": str(REPO_ROOT),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    write_run_status(REPO_ROOT, RUN_ID, stage="eval_export", status="running")

if RUN_EVAL:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    if not MODEL_DIR.exists():
        raise FileNotFoundError(f"model dir not found: {MODEL_DIR}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16)
    model.eval()

    def generate(prompt: str, max_new_tokens: int = 128) -> str:
        toks = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            out = model.generate(**toks, max_new_tokens=max_new_tokens)
        return tokenizer.decode(out[0], skip_special_tokens=True)

    identity_outputs = [generate(p) for p in identity_prompts]
    identity_ok = all(
        ("Lumis-1" in t or "Lumis" in t) and ("Eptesicus" in t)
        for t in identity_outputs
    )
    results["checks"]["identity_correctness"] = {
        "status": "pass" if identity_ok else "fail",
        "outputs": identity_outputs,
    }

    neutral_outputs = [generate(p) for p in neutral_prompts]
    brand_hits = sum(1 for t in neutral_outputs if ("Lumis" in t or "Eptesicus" in t))
    results["checks"]["self_branding_rate"] = {
        "status": "pass",
        "rate": brand_hits / max(len(neutral_outputs), 1),
        "outputs": neutral_outputs,
    }

    no_image_outputs = [generate(p) for p in no_image_prompts]
    hallucination_hits = sum(1 for t in no_image_outputs if "image" not in t.lower() and "can't" not in t.lower())
    results["checks"]["vision_hallucination_on_no_image"] = {
        "status": "pass" if hallucination_hits == 0 else "warn",
        "hallucination_hits": hallucination_hits,
        "outputs": no_image_outputs,
    }

    eval_mm_path = REPO_ROOT / "workspace" / "reports" / "eval_multimodal_prompts.json"
    if eval_mm_path.exists():
        mm_items = json.loads(eval_mm_path.read_text(encoding="utf-8"))
        results["checks"]["multimodal_correctness"] = {
            "status": "manual_review_required",
            "items_loaded": len(mm_items),
        }
    else:
        results["checks"]["multimodal_correctness"] = {
            "status": "skipped",
            "reason": "workspace/reports/eval_multimodal_prompts.json not provided",
        }

parity_pairs_path = REPORTS / "template_parity_pairs.json"
parity_pairs = []
if parity_pairs_path.exists():
    parity_pairs = json.loads(parity_pairs_path.read_text(encoding="utf-8"))

if RUN_EXPORT and not GGUF_DIR.exists():
    raise FileNotFoundError(
        f"RUN_EXPORT is true but GGUF dir is missing: {GGUF_DIR}"
    )

if GGUF_DIR.exists() and parity_pairs:
    results["export_smoke"] = run_export_smoke(GGUF_DIR, parity_pairs)
elif GGUF_DIR.exists():
    results["export_smoke"] = {
        "status": "partial",
        "reason": "GGUF dir exists but template_parity_pairs.json missing",
    }
else:
    results["export_smoke"] = {
        "status": "skipped",
        "reason": "GGUF export directory not found",
    }

out = REPORTS / "export_smoke.json"
out.write_text(json.dumps(results, indent=2), encoding="utf-8")
if run_paths is not None:
    report_copy = run_paths["reports"] / "export_smoke.json"
    report_copy.write_text(json.dumps(results, indent=2), encoding="utf-8")
    checksum_payload = {
        "export_smoke_report": sha256_file(out),
    }
    if GGUF_DIR.exists():
        checksum_payload["gguf_artifacts"] = {
            str(path.relative_to(GGUF_DIR)): sha256_file(path)
            for path in sorted(GGUF_DIR.rglob("*"))
            if path.is_file()
        }
    (run_paths["checksums"] / "artifacts.json").write_text(
        json.dumps(checksum_payload, indent=2),
        encoding="utf-8",
    )
    assessment = assess_eval_export_status(
        results=results,
        run_eval=RUN_EVAL,
        run_export=RUN_EXPORT,
        run_paths=run_paths,
    )
    results["run_status_assessment"] = assessment
    write_run_status(REPO_ROOT, RUN_ID, stage="eval_export", status=assessment["status"], details=results)
    write_run_summary(
        REPO_ROOT,
        RUN_ID,
        "# Eval / Export Summary\n\nThis summary reflects notebook 60 execution.\n\n```json\n"
        + json.dumps(results, indent=2)
        + "\n```",
    )
print(json.dumps(results, indent=2))
print(f"saved: {out}")
